# 5.1 Sampling Techniques

**Goal:** choose a sampling design that matches the population, frame, and available resources.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Sampling Techniques](1_Sampling-Techniques.ipynb)
2. [Sampling Concepts](2_Sampling-Concepts.ipynb)
3. [Estimation Theory](3_Estimation-Theory.ipynb)
4. [Confidence Intervals](4_Confidence-Intervals.ipynb)
5. [Hypothesis Testing](5_Hypothesis-Testing.ipynb)
6. [Statistical Errors](6_Statistical-Errors.ipynb)
7. [Statistical Tests](7_Statistical-Tests.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Simple Random Sampling

A probability sample uses known, positive selection probabilities for eligible units. In **simple random sampling (SRS) without replacement**, every size-n subset of a frame of N units is equally likely; each unit has inclusion probability n/N. A random draw from an incomplete frame still does not represent omitted units automatically.

**Worked example:** sample 10 of 100 user IDs. A random number generator chooses the IDs, but deciding that the frame includes only recently active users is a separate design decision. SRS is easy to analyze but can miss small important groups by chance. Define the independent unit (user, household, transaction) and preserve its identifier.

In [2]:
frame=np.arange(100)
srs=rng.choice(frame,size=10,replace=False)
print("SRS user IDs:",srs)
assert len(np.unique(srs))==10

SRS user IDs: [ 2 35 16 60 77 34  7 44 36 63]


## 2. Systematic Sampling

For N=nk, choose a uniform random start from 0,…,k−1 and then every k-th frame entry. Each unit has probability 1/k, but not every subset is possible. For other N,n, specify an appropriate interval or circular design instead of assuming the simple formula is exact.

**Example:** N=100,n=10 gives k=10. Start 3 selects 3,13,…,93. Periodicity can undermine the design: if every tenth row belongs to a particular shift, one systematic sample may contain only that shift. Random starts do not remove all ordering problems; assess frame order and design-based variance.

In [3]:
k=10
start=int(rng.integers(k))
systematic=frame[start::k]
print("Random start:",start,"systematic sample:",systematic)

Random start: 2 systematic sample: [ 2 12 22 32 42 52 62 72 82 92]


## 3. Stratified Sampling

Partition the population into nonoverlapping exhaustive **strata**, then sample independently within every stratum. Proportional allocation uses n_h≈nN_h/N. Disproportionate allocation can deliberately oversample rare groups, but population summaries must restore population weights.

For stratum sample means, estimate the overall mean by $\hat\mu=\sum_h(N_h/N)\bar x_h$. Under independent SRS within strata, its variance is $\sum_h(N_h/N)^2(1-n_h/N_h)S_h^2/n_h$, where $S_h^2$ is the stratum finite-population variance with denominator $N_h-1$.

**Example:** a population is 90% basic and 10% premium. Equal sample counts do not justify equal population weights; means 2 and 8 give overall estimate 0.9(2)+0.1(8)=2.6, not 5.

In [4]:
strata={"basic":np.arange(90),"premium":np.arange(90,100)}
selected={h:rng.choice(ids,size=5,replace=False) for h,ids in strata.items()}
print("Stratified sample:",selected)
print("Population-weighted mean:",0.9*2+0.1*8,"unweighted group mean:",(2+8)/2)

Stratified sample: {'basic': array([84, 14, 25, 64, 81]), 'premium': array([95, 99, 93, 94, 96])}
Population-weighted mean: 2.6 unweighted group mean: 5.0


## 4. Cluster Sampling and Multistage Sampling

In **cluster sampling**, select some natural groups such as schools or stores. A one-stage design observes all eligible units in selected clusters. Clusters differ from strata: stratification samples within every stratum; clustering selects only some clusters. Similarity within clusters usually reduces effective information relative to independent individuals.

**Multistage sampling** selects in stages: districts, then schools within districts, then students within schools. Final selection probabilities multiply relevant conditional stage probabilities. Weights and uncertainty must reflect the complete design. Selecting the same number of students from differently sized schools can create unequal individual probabilities.

**Example:** sample 2 of 10 equal-sized schools and all 20 students per selected school. There are 40 rows but only two sampled clusters. A row-level IID standard error can be seriously optimistic. If a common intracluster correlation approximation applies, design effect is about $1+(m-1)\rho$, not a universal correction for every design.

In [5]:
schools={i:np.arange(20)+20*i for i in range(10)}
chosen=rng.choice(list(schools),size=2,replace=False)
cluster_sample=np.concatenate([schools[i] for i in chosen])
multistage=np.concatenate([rng.choice(schools[i],size=5,replace=False) for i in chosen])
print("Schools:",chosen,"one-stage rows:",len(cluster_sample),"two-stage rows:",len(multistage))
print("Illustrative design effect, m=20 and rho=.1:",1+19*.1)

Schools: [1 2] one-stage rows: 40 two-stage rows: 10
Illustrative design effect, m=20 and rho=.1: 2.9000000000000004


## 5. Sampling with Replacement and Sampling without Replacement

With replacement, a selected unit returns to the pool; repeated IDs can occur and uniform successive draws can be independent. Without replacement, IDs cannot repeat and draw outcomes are dependent. Replacement is also a design choice in bootstrap resampling, which samples from observed data rather than directly from the population.

For SRS without replacement, if finite-population variance uses denominator N, $\operatorname{Var}(\bar X)=\sigma_N^2(N-n)/[n(N-1)]$. Equivalently, with variance $S^2$ using N−1, it is $(1-n/N)S^2/n$. State the variance convention when quoting a finite-population correction.

In [6]:
with_replacement=rng.choice(np.arange(5),size=12,replace=True)
print("Replacement sample:",with_replacement)
N,n=100,20
print("Finite-population SE multiplier relative to sigma_N/sqrt(n):",np.sqrt((N-n)/(N-1)))

Replacement sample: [1 4 1 4 4 4 3 2 3 1 1 2]
Finite-population SE multiplier relative to sigma_N/sqrt(n): 0.8989331499509894


## AI/ML application and interpretation

For an ML audit, stratify by important user groups when subgroup precision matters, preserve user/cluster identifiers, and report weighted population summaries when inclusion probabilities differ. Split training and evaluation at the relevant entity level. Sampling design can improve coverage, but cannot manufacture unavailable labels or repair an invalid frame.

## Practice

1. Contrast stratified and cluster sampling.
2. A population is 80% group A and 20% B; estimated means are 5 and 15. Estimate the population mean.
3. Why can systematic sampling be risky for a periodically ordered list?
4. Does collecting 100 rows from two households give 100 independent household observations?
5. Give a valid two-stage design and explain the role of weights.

## Solutions

1. Sample from every stratum; select only some clusters, then observe all or a subsample within those selected.
2. 0.8(5)+0.2(15)=7, provided within-group samples support those estimates.
3. The sampling interval can align with a repeating process and poorly represent other positions.
4. No. Repeated observations within a household may be strongly dependent.
5. Select schools with known probabilities, then students within schools. A student's overall inclusion probability depends on both stages; inverse-probability weights restore the intended population contribution.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](2_Sampling-Concepts.ipynb).